# VoxEntity-EU — evaluation notebook

Receives the frozen audio bundle, downloads pinned weights, transcribes per model and stores predictions. Gold is never mounted here: scoring happens in a separate step with the gold-side mapping.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[asr-whisper]'], cwd='../..').returncode)

In [ ]:
# Whisper turbo run over the exported inputs (GPU)
from voxentity_eu.cli import main as cli_main
print(cli_main(['run', '--inputs', 'data/inputs', '--system', 'whisper-turbo', '--out', 'runs/whisper-turbo', '--resume']))

In [ ]:
# Voxtral Mini runs on Kaggle (scripts/kaggle/voxtral_run.py); predictions are
# retrieved with `kaggle kernels output gjusev/voxentity-eu-voxtral-run` and
# placed in runs/voxtral-mini/. Voxtral is a separate family, reported as its own system.

In [ ]:
# Slot extraction over transcripts (track B) + gold->extractor ceiling + agent (track D)
import sys; sys.path.insert(0, 'scripts')
from pathlib import Path
from run_tracks import build_asr_slots_run, build_gold_extractor_run, run_reference_agent
print(build_asr_slots_run(Path('runs/whisper-turbo'), Path('data/inputs'), Path('runs/whisper-turbo-slots')))
print(build_gold_extractor_run(Path('data/generated/gold/scenarios.jsonl'), Path('data/generated/audio_manifest.jsonl'), Path('data/inputs'), Path('runs/gold-extractor')))

In [ ]:
# Human control (FLEURS) + WER/CER, separate from synthetic scores
print(cli_main(['human-control', '--source-lock', 'research/artifacts.json', '--out', 'data/human_control']))